# Práctica – Embeddings Semánticos con Sentence Transformers
## UNR · TUIA · Procesamiento de Lenguaje Natural – Unidad 2

En esta práctica trabajamos con **modelos de embeddings semánticos** densos
usando el modelo liviano `intfloat/multilingual-e5-small` (384 dimensiones, ~118 MB).

A diferencia de los métodos frecuentistas (TF-IDF, Count), estos embeddings
capturan el **significado semántico** del texto: dos frases con palabras
completamente distintas pero significados similares tendrán vectores cercanos.

| Parte | Tema |
|-------|------|
| 1 | Carga del dataset y preparación |
| 2 | Vectorización y almacenamiento en ZIP |
| 3 | Importación de embeddings desde ZIP |
| 4 | Búsqueda semántica de libros |
| 5 | Sistema de recomendación "Libros similares" |
| 6 | Análisis de similitud entre géneros |
| 7 | Clustering con K-Means |
| 8 | Visualización 2D/3D del espacio semántico |
| 9 | Evaluación comparativa: E5 vs TF-IDF |

> **Flujo de trabajo**: La Parte 2 vectoriza todo el dataset y guarda los embeddings
> en un archivo ZIP. A partir de la Parte 3, se importan desde el ZIP para no
> tener que re-vectorizar cada vez que se abre el notebook.


In [ ]:
# ── Instalaciones (ejecutar una vez) ──────────────────────────────────────
# !pip install sentence-transformers scikit-learn numpy pandas matplotlib seaborn

# ── Imports globales ──────────────────────────────────────────────────────
import re, os, zipfile, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score, adjusted_rand_score
from collections import Counter

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100


---
# Parte 1 – Carga y Preparación del Dataset

Usamos el dataset de **Lectulandia** (~62k libros). Filtramos los que tienen
sinopsis y limpiamos el texto para la vectorización.


In [ ]:
# ── Cargar dataset ────────────────────────────────────────────────────────
df_raw = pd.read_csv("data/lectulandia_books.csv")
print(f"Dataset original: {len(df_raw):,} libros, {df_raw.columns.tolist()}")
print(f"Libros sin sinopsis: {df_raw['sinapsis'].isna().sum():,}")

# Filtrar libros con sinopsis
df = df_raw.dropna(subset=["sinapsis"]).copy()
df = df[df["sinapsis"].str.strip().str.len() > 50].reset_index(drop=True)
print(f"Libros con sinopsis válida (>50 chars): {len(df):,}")

# Limpieza básica del texto
def limpiar_texto(texto):
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["sinopsis_clean"] = df["sinapsis"].apply(limpiar_texto)

# Extraer género principal (el primero de la lista)
df["genero_principal"] = df["generos"].fillna("Sin género").apply(
    lambda x: x.split(" - ")[0].strip()
)

# Estadísticas rápidas
print(f"\nLongitud promedio de sinopsis: {df['sinopsis_clean'].str.len().mean():.0f} caracteres")
print(f"Géneros únicos (principal): {df['genero_principal'].nunique()}")
print(f"\nTop 10 géneros:")
print(df["genero_principal"].value_counts().head(10).to_string())

Dataset original: 62,279 libros, ['url', 'titulo', 'autor', 'autor_url', 'sinapsis', 'imagen_url', 'generos']
Libros sin sinopsis: 14,449
Libros con sinopsis válida (>50 chars): 47,819

Longitud promedio de sinopsis: 929 caracteres
Géneros únicos (principal): 100

Top 10 géneros:
genero_principal
Novela               9363
Intriga              5304
Histórico            3604
Drama                3305
Ensayo               2715
Aventuras            2506
Ciencia ficción      2440
Fantástico           2340
Ciencias sociales    2197
Crónica              2083


---
# Parte 2 – Vectorización con `intfloat/multilingual-e5-small` y Guardado en ZIP

## Sobre el modelo

`intfloat/multilingual-e5-small` es un modelo de embeddings multilingüe entrenado
con la técnica E5 (*EmbEddings from tExtual data with Text-to-text format*).

| Propiedad | Valor |
|-----------|-------|
| Dimensiones | 384 |
| Parámetros | ~118M |
| Idiomas | 100+ (incluye español) |
| Tamaño en disco | ~450 MB |
| Velocidad | Rápido en CPU |

**Importante**: E5 usa un prefijo para indicar el tipo de texto:
- `"query: "` para consultas de búsqueda
- `"passage: "` para documentos/pasajes a indexar

Esto mejora la calidad de los embeddings significativamente.

> **Esta celda tarda ~5-10 minutos en CPU para el dataset completo (~48k libros).** Los embeddings se guardan en un ZIP
> para no tener que repetir este paso. Si ya tenés el ZIP, saltá a la Parte 3.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELDA DE VECTORIZACIÓN — Ejecutar UNA vez, luego usar el ZIP
# ══════════════════════════════════════════════════════════════════════════
from sentence_transformers import SentenceTransformer
import time

MODEL_NAME = "intfloat/multilingual-e5-small"
model = SentenceTransformer(MODEL_NAME)

# Preparar textos con el prefijo "passage:" que usa E5
textos_para_encode = ["passage: " + t for t in df["sinopsis_clean"].tolist()]

print(f"Vectorizando {len(textos_para_encode):,} sinopsis con {MODEL_NAME}...")
print(f"Esto puede tardar ~5-10 minutos en CPU...")

t0 = time.time()
embeddings = model.encode(
    textos_para_encode,
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True
)
elapsed = time.time() - t0

print(f"\nVectorización completada en {elapsed:.1f}s")
print(f"Shape de embeddings: {embeddings.shape}")
print(f"Tipo: {embeddings.dtype}")
print(f"Norma L2 promedio: {np.linalg.norm(embeddings, axis=1).mean():.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23979.16it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorizando 47,819 sinopsis con intfloat/multilingual-e5-small...
Esto puede tardar ~5-10 minutos en CPU...


Batches: 100%|██████████| 748/748 [00:46<00:00, 16.24it/s]


Vectorización completada en 46.7s
Shape de embeddings: (47819, 384)
Tipo: float32
Norma L2 promedio: 1.0000


### Guardar embeddings y metadata en ZIP

Guardamos en un archivo ZIP:
- `embeddings.npy` — la matriz numpy de embeddings (shape: N x 384)
- `metadata.csv` — información de cada libro (título, autor, género, sinopsis)
- `config.json` — configuración del modelo y parámetros usados

Así cualquier celda posterior puede importar todo sin necesidad de re-vectorizar.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# GUARDAR EMBEDDINGS EN ZIP
# ══════════════════════════════════════════════════════════════════════════
import json

ZIP_PATH = "data/embeddings_e5_small.zip"

# Preparar metadata
df_meta = df[["titulo", "autor", "sinopsis_clean", "genero_principal", "generos"]].copy()
df_meta.columns = ["titulo", "autor", "sinopsis", "genero_principal", "generos"]

# Configuración
config = {
    "model_name": MODEL_NAME,
    "embedding_dim": int(embeddings.shape[1]),
    "n_documents": int(embeddings.shape[0]),
    "normalized": True,
    "prefix_used": "passage: ",
    "dtype": str(embeddings.dtype),
}

# Guardar en ZIP
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    # Embeddings como .npy
    buf = io.BytesIO()
    np.save(buf, embeddings)
    zf.writestr("embeddings.npy", buf.getvalue())

    # Metadata como CSV
    csv_buf = io.StringIO()
    df_meta.to_csv(csv_buf, index=False)
    zf.writestr("metadata.csv", csv_buf.getvalue())

    # Config como JSON
    zf.writestr("config.json", json.dumps(config, indent=2, ensure_ascii=False))

file_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Guardado en: {ZIP_PATH}")
print(f"   Tamaño: {file_size_mb:.1f} MB")
print(f"   Contenido:")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for info in zf.infolist():
        print(f"     - {info.filename}: {info.file_size / 1024:.1f} KB")

Guardado en: data/embeddings_e5_small.zip
   Tamaño: 81.8 MB
   Contenido:
     - embeddings.npy: 71728.6 KB
     - metadata.csv: 47969.9 KB
     - config.json: 0.2 KB


---
# Parte 3 – Importar Embeddings desde ZIP

> **A partir de acá, todas las celdas usan los embeddings pre-calculados.**
> No hace falta tener `sentence-transformers` instalado ni el modelo descargado
> para resolver los ejercicios — solo el archivo ZIP.

Si no generaste el ZIP, podés descargarlo desde [Google Drive](https://drive.google.com/file/d/1SxsCy9airq_1OaNKFVUu_SB7HGSTe_gk/view?usp=sharing) y colocarlo en `data/embeddings_e5_small.zip`.

Ejecutá la siguiente celda para cargar embeddings y metadata.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# IMPORTAR EMBEDDINGS DESDE ZIP — Ejecutar siempre
# ══════════════════════════════════════════════════════════════════════════
import zipfile, io, json
import numpy as np
import pandas as pd

ZIP_PATH = "data/embeddings_e5_small.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    # Cargar embeddings
    with zf.open("embeddings.npy") as f:
        embeddings = np.load(io.BytesIO(f.read()))

    # Cargar metadata
    with zf.open("metadata.csv") as f:
        df_libros = pd.read_csv(io.BytesIO(f.read()))

    # Cargar config
    with zf.open("config.json") as f:
        config = json.loads(f.read())

print(f"Embeddings cargados desde ZIP")
print(f"   Modelo: {config['model_name']}")
print(f"   Documentos: {embeddings.shape[0]:,}")
print(f"   Dimensiones: {embeddings.shape[1]}")
print(f"   Normalizados: {config['normalized']}")
print(f"\nPrimeras filas del dataset:")
df_libros.head()

Embeddings cargados desde ZIP
   Modelo: intfloat/multilingual-e5-small
   Documentos: 47,819
   Dimensiones: 384
   Normalizados: True

Primeras filas del dataset:


,titulo,autor,sinopsis,genero_principal,generos
0,Tierra feroz,Jota Quijorna,La oscuridad de un alma herida es el hogar de ...,Intriga,Intriga - Novela
1,Tehanu,Ursula K. Le Guin,"El mal medra, y la magia se ha pervertido. En ...",Fantástico,Fantástico - Novela
2,El niño del taxi,Sylvain Prudhomme,"Durante el funeral de su abuelo, Simon descubr...",Histórico,Histórico - Novela
3,James Bond 007: El juego de rol,Gerard Christopher Klug,ATRÉVETE A VIVIR EN EL FASCINANTE MUNDO DE JAM...,Juegos,Juegos - Manuales y cursos - Referencia
4,El retorno del cuervo,Alissa Brontë,"Tras varios años alejado del que fue su hogar,...",Histórico,Histórico - Novela - Romántico


---
# Parte 4 – Búsqueda Semántica de Libros

Los embeddings semánticos permiten buscar libros por **significado**, no por
palabras clave exactas.

### Ejercicio 1 – Búsqueda Semántica por Texto

Implementá `buscar_libros` que reciba una query en lenguaje natural y
devuelva los N libros más relevantes usando similitud coseno.

**Recordá** que E5 usa el prefijo `"query: "` para las consultas.

In [ ]:
def buscar_libros(query, embeddings, df_libros, model, n=5):
    """
    TODO:
    1. Codificar la query con model.encode() usando prefijo "query: "
    2. Calcular cosine_similarity contra todos los embeddings
    3. Retornar los top-N como DataFrame (titulo, autor, genero_principal, score)
    """
    pass


queries_test = [
    "historia de amor en tiempos de guerra",
    "aventuras de fantasía con dragones y magia",
    "divulgación científica sobre el universo",
]

# Descomentar cuando implementes:
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("intfloat/multilingual-e5-small")
# for q in queries_test:
#     print(f"\n «{q}»")
#     print(buscar_libros(q, embeddings, df_libros, model).to_string(index=False))

---
# Parte 5 – Libros Similares (sin modelo)

Dado un libro del dataset, encontrar los más parecidos usando solo los
embeddings precalculados. **No requiere cargar el modelo.**

### Ejercicio 2 – Encontrar Libros Similares

Implementá `libros_similares` que dado el índice de un libro retorne
los N más parecidos (excluyéndose a sí mismo).

In [ ]:
def libros_similares(idx, embeddings, df_libros, n=5):
    """
    TODO:
    1. Obtener el embedding del libro en posición idx
    2. Calcular cosine_similarity contra todos los embeddings
    3. Excluir el propio libro
    4. Retornar DataFrame con: titulo, autor, genero_principal, score
    """
    pass

# libro = df_libros.iloc[0]
# print(f"Similares a «{libro['titulo']}»:")
# print(libros_similares(0, embeddings, df_libros))

---
# Parte 6 – Similitud entre Géneros

Calculando el **embedding promedio** de cada género podemos medir qué tan
parecidos son los géneros entre sí.

**Importante**: Antes de calcular centroides, hay que **centrar los embeddings**
restando la media global. Sin esto, todos los centroides quedan con similitud
~0.98 porque comparten un componente común ("es una sinopsis de libro en español").

### Ejercicio 3 – Heatmap de Similitud entre Géneros

1. **Centrar** los embeddings (restar la media global)
2. Filtrar los géneros con al menos 1000 libros
3. Calcular el centroide (promedio de embeddings centrados) de cada género
4. Armar la matriz de similitud coseno entre centroides
5. Mostrar como heatmap

In [ ]:
def similitud_generos(embeddings, df_libros, min_libros=1000):
    """
    TODO:
    1. Centrar embeddings: restar la media global (embeddings - embeddings.mean(axis=0))
    2. Filtrar géneros con >= min_libros
    3. Calcular centroide por género (promedio de embeddings centrados, normalizar)
    4. Calcular cosine_similarity entre centroides
    5. Graficar heatmap con sns.heatmap
    """
    pass

# similitud_generos(embeddings, df_libros)

---
# Parte 7 – Clustering con K-Means

¿Los embeddings agrupan naturalmente los libros por género?
Probemos con K-Means y visualicemos con PCA.

### Ejercicio 4 – Clustering sobre Embeddings

1. Filtrar los **top 5 géneros** más frecuentes
2. Aplicar K-Means (K=5) sobre los embeddings
3. Calcular **Silhouette Score**
4. Visualizar en 2D con PCA, coloreando por cluster

In [ ]:
def clustering_embeddings(embeddings, df_libros, n_generos=5):
    """
    TODO:
    1. Filtrar top n_generos más frecuentes
    2. Aplicar KMeans(n_clusters=n_generos) sobre sus embeddings
    3. Calcular silhouette_score
    4. PCA a 2D y scatter plot coloreado por cluster
    """
    pass

# clustering_embeddings(embeddings, df_libros)

---
# Parte 8 – Visualización del Espacio Semántico

Reducimos 384 dimensiones a 2D para ver cómo se distribuyen los libros.

### Ejercicio 5 – Mapa Semántico con PCA

1. Filtrar los **top 8 géneros**
2. Aplicar PCA a 2D
3. Scatter plot coloreado por género con leyenda

In [ ]:
def mapa_semantico(embeddings, df_libros, n_generos=8):
    """
    TODO:
    1. Filtrar top n_generos
    2. PCA a 2D
    3. Scatter plot con colores por género y leyenda
    """
    pass

# mapa_semantico(embeddings, df_libros)

---
# Parte 9 – Comparativa: E5 vs TF-IDF

¿Qué tan distintos son los resultados de una búsqueda semántica (E5)
vs una búsqueda por palabras clave (TF-IDF)?

### Ejercicio 6 – E5 vs TF-IDF en Búsqueda

1. Vectorizar las sinopsis con `TfidfVectorizer`
2. Para 5 queries, buscar top-5 con ambos métodos
3. Contar cuántos resultados tienen en común (overlap)

In [ ]:
def comparar_e5_tfidf(embeddings, df_libros, model):
    """
    TODO:
    1. Crear TfidfVectorizer y vectorizar sinopsis
    2. Para cada query: buscar top-5 con E5 y con TF-IDF
    3. Calcular overlap (cuántos libros en común de los 5)
    4. Imprimir resultados lado a lado
    """
    pass

# comparar_e5_tfidf(embeddings, df_libros, model)

---
# Resumen

| Aspecto | TF-IDF | E5-small (Sentence Transformer) |
|---------|--------|---------------------------------|
| **Tipo de vector** | Sparse, alta dimensión | Denso, 384 dims |
| **Semántica** | Léxica (palabras exactas) | Contextual (significado) |
| **Multilingüe** | No | Sí (100+ idiomas) |
| **Velocidad** | Muy rápido | Moderado |
| **Mejor para** | Búsqueda por keywords | Búsqueda por significado |

### Flujo recomendado

1. **Vectorizar una vez** con el modelo E5
2. **Guardar en ZIP** para reutilizar
3. **Importar embeddings** sin necesidad del modelo
4. Solo cargar el modelo para **queries nuevas**